## Topics Covered
- [Sementic Chunking](#sementic-chunking)
- [RAG Using Sementic Chunking (Modular Code)](#rag-using-sementic-chunking-modular-code)
- [Sementic Chunking Using Langchain](#sementic-chunking-using-langchain)

### Sementic Chunking
- SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.

- It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [5]:
## Sample text
text="""
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

In [9]:
### step 1 : Split into sentences
sentences=[s.strip() for s in text.split("\n") if s.strip()]

### step 2: Embed each setence
embeddings=model.encode(sentences)

### step 3: Initialize parameters
threshold = 0.7  # control chunk tightness
chunks = []
current_chunk=[sentences[0]]

### step 4: Semantic grouping based on threshold
for i in range(1, len(sentences)):
    sim = cosine_similarity(
        embeddings[i - 1].reshape(1, -1),
        embeddings[i].reshape(1, -1)
    )[0][0]

    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk=[sentences[i]]

# Append the last chunk
chunks.append(" ".join(current_chunk))

# Output the chunks
print("\n📌 Semantic Chunks:")
for idx, chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}:\n{chunk}")


📌 Semantic Chunks:

Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
You can create chains, agents, memory, and retrievers.

Chunk 3:
The Eiffel Tower is located in Paris.

Chunk 4:
France is a popular tourist destination.


### RAG Using Sementic Chunking (Modular Code)

In [25]:
# import libraries
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from pinecone import Pinecone
from pinecone import ServerlessSpec

from langchain_core.documents import Document
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain.schema.runnable import RunnableLambda, RunnableMap
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_pinecone import PineconeVectorStore

In [19]:
load_dotenv()

True

In [20]:
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

# set environment variable for OpenAI API key
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

# set environment variable for Groq API key
if groq_key is not None:
    os.environ["GROQ_API_KEY"] = groq_key
else:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

In [21]:
### Custom Semantic Chunker With Threshold
class ThresholdSemanticChunker(object):

    def __init__(self, model_name='all-MiniLM-L6-v2', threshold=0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def _split(self, text:str):
        sentences = [s.strip() for s in text.split(".") if s.strip()]
        embeddings = self.model.encode(sentences)

        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity(
                embeddings[i - 1].reshape(1, -1),
                embeddings[i].reshape(1, -1)
            )[0][0]

            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]

        chunks.append(". ".join(current_chunk) + ".")
        return chunks
    
    def split_documents(self,docs):
        result=[]
        for doc in docs:
            for chunk in self._split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))

        return result

In [22]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [23]:
### Chunking
chunker=ThresholdSemanticChunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks

[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [26]:
### VectorStore (Pinecone)
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)

index_name = "rag-mrinmoy"

if not pc.has_index(index_name):
    index = pc.create_index(
        name = index_name,
        dimension = 1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [27]:
vectorstore = PineconeVectorStore(
    index=index,
    embedding=openai_embeddings
)

In [28]:
vectorstore.add_documents(chunks)

['232c71d4-00b9-44fd-af81-f532b06d1dd9',
 '5a389c4b-d040-4183-bdbb-a2f13e1498a2',
 'f6a0f2f4-5510-42a7-9430-b5fae11e0858',
 'd0677da6-5f11-44a4-9eac-30761576ecdd']

In [29]:
# convert vectorstore to retriver
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold", 
    search_kwargs={"k": 3, "score_threshold": 0.5}
)


In [34]:
## Prompt Template
template = """Answer the question based on the following context:

{context}

Question: {question}
Answer:
"""

prompt = PromptTemplate.from_template(template)

In [35]:
llm=init_chat_model(model="groq:gemma2-9b-it",temperature=0.4)

In [36]:
### LCEL Chain With retrieval
rag_chain=(
    RunnableMap(
        {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],  
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [37]:
# Run Query
query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

print(result)

According to the provided context, LangChain is a framework for building applications with LLMs. 



### Sementic Chunking Using Langchain

In [38]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain.document_loaders import TextLoader

In [39]:
## Load the documents
loader=TextLoader("langchain_intro.txt")
docs=loader.load()

## Initialize embedding model
embedding=OpenAIEmbeddings()

## Create the semantic chunker
chunker=SemanticChunker(embedding)

## Split the documents
chunks=chunker.split_documents(docs)

## Result

for i,chunk in enumerate(chunks):
    print(f"\n chunk {i+1}:\n{chunk.page_content}")


 chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

 chunk 2:
You can create chains, agents, memory, and retrievers. The Eiffel Tower is located in Paris. France is a popular tourist destination.
